## 06 - Sentiment Analysis with VADER

This notebook scores each earnings call transcript for sentiment
using VADER - a pre-built sentiment analysis tool.

### What we measure?
- Positive score: how much confident/positive language
- Negative score: how much uncertain/negative language  
- Compound score: overall sentiment from -1 to +1

### Why VADER first?
VADER is simple, fast and works well for general text.
In the next notebook we'll use FinBERT which is specifically
trained on financial language for more accurate results.

In [1]:
# Import libraries and download VADER
import nltk
import sqlite3
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADDER lexicon - only needed once
nltk.download('vader_lexicon')

# Initialize VADER
sia = SentimentIntensityAnalyzer()

print("VADER ready!")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...


VADER ready!


In [4]:
# Test VADER on simple sentences first
test_sentences = [ 
    "We had an exceptional quarter with records breaking revenue.",
    "We are uncertain about the outlook for the coming quarter.",
    "Given the lack of visibility, we will not be issuing guidance.",
    "We are optimistic about our long term growth prospects.",
    "The wider than usual revenue range comprehends uncertainty."
]

for sentence in test_sentences:
    scores = sia.polarity_scores(sentence)
    print(f"Text: {sentence[:60]}...")
    print(f"Scores: {scores}")
    print()

Text: We had an exceptional quarter with records breaking revenue....
Scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Text: We are uncertain about the outlook for the coming quarter....
Scores: {'neg': 0.196, 'neu': 0.804, 'pos': 0.0, 'compound': -0.296}

Text: Given the lack of visibility, we will not be issuing guidanc...
Scores: {'neg': 0.187, 'neu': 0.813, 'pos': 0.0, 'compound': -0.3182}

Text: We are optimistic about our long term growth prospects....
Scores: {'neg': 0.0, 'neu': 0.458, 'pos': 0.542, 'compound': 0.7269}

Text: The wider than usual revenue range comprehends uncertainty....
Scores: {'neg': 0.255, 'neu': 0.745, 'pos': 0.0, 'compound': -0.34}



In [5]:
# Connect to database and score all transcripts
conn = sqlite3.connect("../data/earnings_research.db")

# Load all transcripts
transcripts = pd.read_sql("""
    SELECT id, ticker, quarter, date, period, raw_text
    FROM transcripts
    ORDER BY date
""", conn)

print(f"Loaded {len(transcripts)} transcripts")
print(transcripts[["ticker", "quarter", "period"]])

Loaded 14 transcripts
   ticker  quarter          period
0    AAPL  Q1_2020       Pre-COVID
1     JNJ  Q1_2020       Pre-COVID
2     JPM  Q1_2020       Pre-COVID
3    MSFT  Q1_2020       Pre-COVID
4    AAPL  Q2_2020     COVID Crash
5    AMZN  Q1_2020       Pre-COVID
6     XOM  Q1_2020       Pre-COVID
7     JPM  Q2_2020     COVID Crash
8    MSFT  Q4_2022  Rate Hike Peak
9    AAPL  Q4_2022  Rate Hike Peak
10   MSFT  Q2_2023         AI Boom
11   NVDA  Q2_2023         AI Boom
12   AAPL  Q1_2025    Tariff Shock
13   AMZN  Q1_2025    Tariff Shock


In [7]:
# Score all 14 transcripts with VADER
results =[]

for _, row in transcripts.iterrows():
    # Get sentiment scores for the full transcript
    scores = sia.polarity_scores(row["raw_text"])

    # Store results
    results.append({
        "ticker": row["ticker"],
        "quarter": row["quarter"],
        "date": row["date"],
        "period": row["period"],
        "positive": round(scores["pos"], 4),
        "negative": round(scores["neg"], 4),
        "neutral": round(scores["neu"], 4),
        "compound": round(scores["compound"], 4)
    })

    print(f"✓ {row['ticker']} {row['quarter']} → compound: {round(scores['compound'], 4)}")

print("\nAll transcripts score!")

✓ AAPL Q1_2020 → compound: 0.9999
✓ JNJ Q1_2020 → compound: 1.0
✓ JPM Q1_2020 → compound: 0.9997
✓ MSFT Q1_2020 → compound: 0.9999
✓ AAPL Q2_2020 → compound: 1.0
✓ AMZN Q1_2020 → compound: 0.9893
✓ XOM Q1_2020 → compound: 0.9999
✓ JPM Q2_2020 → compound: 0.9992
✓ MSFT Q4_2022 → compound: 1.0
✓ AAPL Q4_2022 → compound: 1.0
✓ MSFT Q2_2023 → compound: 1.0
✓ NVDA Q2_2023 → compound: 1.0
✓ AAPL Q1_2025 → compound: 1.0
✓ AMZN Q1_2025 → compound: 1.0

All transcripts score!


In [8]:
# Score sentence by sentence for more nuanced results
from nltk.tokenize import sent_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

results_v2 = []

for _, row in transcripts.iterrows():
    # Split transcript into sentences
    sentences = sent_tokenize(row["raw_text"])
    
    # Score each sentence
    sentence_scores = []
    for sentence in sentences:
        score = sia.polarity_scores(sentence)["compound"]
        sentence_scores.append(score)
    
    # Average the scores
    avg_compound = round(sum(sentence_scores) / len(sentence_scores), 4)
    max_positive = round(max(sentence_scores), 4)
    min_negative = round(min(sentence_scores), 4)
    
    results_v2.append({
        "ticker": row["ticker"],
        "quarter": row["quarter"],
        "date": row["date"],
        "period": row["period"],
        "avg_compound": avg_compound,
        "max_positive": max_positive,
        "min_negative": min_negative,
        "sentence_count": len(sentence_scores)
    })
    
    print(f"✓ {row['ticker']} {row['quarter']} → avg: {avg_compound} ({len(sentences)} sentences)")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


✓ AAPL Q1_2020 → avg: 0.2968 (150 sentences)
✓ JNJ Q1_2020 → avg: 0.2844 (441 sentences)
✓ JPM Q1_2020 → avg: 0.1794 (178 sentences)
✓ MSFT Q1_2020 → avg: 0.3113 (211 sentences)
✓ AAPL Q2_2020 → avg: 0.3193 (176 sentences)
✓ AMZN Q1_2020 → avg: 0.1272 (53 sentences)
✓ XOM Q1_2020 → avg: 0.1838 (220 sentences)
✓ JPM Q2_2020 → avg: 0.0901 (220 sentences)
✓ MSFT Q4_2022 → avg: 0.2529 (503 sentences)
✓ AAPL Q4_2022 → avg: 0.2228 (504 sentences)
✓ MSFT Q2_2023 → avg: 0.2492 (526 sentences)
✓ NVDA Q2_2023 → avg: 0.1511 (530 sentences)
✓ AAPL Q1_2025 → avg: 0.2703 (513 sentences)
✓ AMZN Q1_2025 → avg: 0.278 (188 sentences)


In [9]:
# Save sentiment scores to database
df_results_v2 = pd.DataFrame(results_v2)

df_results_v2.to_sql("sentiment_scores", conn, if_exists="replace", index=False)
conn.commit()

print("Sentiment scores saved to database!")
print(f"\nTotal records: {len(df_results_v2)}")
print("\nFull results:")
print(df_results_v2[["ticker", "quarter", "period", "avg_compound"]].to_string(index=False))

Sentiment scores saved to database!

Total records: 14

Full results:
ticker quarter         period  avg_compound
  AAPL Q1_2020      Pre-COVID        0.2968
   JNJ Q1_2020      Pre-COVID        0.2844
   JPM Q1_2020      Pre-COVID        0.1794
  MSFT Q1_2020      Pre-COVID        0.3113
  AAPL Q2_2020    COVID Crash        0.3193
  AMZN Q1_2020      Pre-COVID        0.1272
   XOM Q1_2020      Pre-COVID        0.1838
   JPM Q2_2020    COVID Crash        0.0901
  MSFT Q4_2022 Rate Hike Peak        0.2529
  AAPL Q4_2022 Rate Hike Peak        0.2228
  MSFT Q2_2023        AI Boom        0.2492
  NVDA Q2_2023        AI Boom        0.1511
  AAPL Q1_2025   Tariff Shock        0.2703
  AMZN Q1_2025   Tariff Shock        0.2780


In [10]:
# Close database connection
conn.close()
print("Database connection closed.")

Database connection closed.
